# Stage 04: Localised (span-level) PCL detection (Part 3)

Utilises task2 dataset span text to get pure localised signals of where in paragh pcl is actually occuring, so the model can aggreagate these local signals and global context for more accurate paragraph-level predictions

## Imports & Dataset utilities

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

def _find_repo_root(start: Path) -> Path:
    start = start.resolve()
    candidates = [start, *start.parents]
    for p in candidates:
        if (p / "data" / "raw").exists() and (p / "data" / "splits").exists():
            return p
    raise FileNotFoundError("Could not locate repo root containing data/raw and data/splits")

ROOT = _find_repo_root(Path.cwd())

RAW_TASK1 = ROOT / "data" / "raw" / "dontpatronizeme_pcl.tsv"
RAW_TASK2 = ROOT / "data" / "raw" / "dontpatronizeme_categories.tsv"
TRAIN_SPLIT = ROOT / "data" / "splits" / "train_semeval_parids-labels.csv"
DEV_SPLIT = ROOT / "data" / "splits" / "dev_semeval_parids-labels.csv"

# ---- Task 1 paragraphs (binary labels) ----
pcl_df = pd.read_csv(RAW_TASK1, sep="\t", header=None, skiprows=4)
pcl_df.columns = ["par_id", "art_id", "keyword", "country_code", "text", "label_0to4"]
pcl_df["label_bin"] = (pcl_df["label_0to4"] >= 2).astype(int)

train_ids = pd.read_csv(TRAIN_SPLIT)[["par_id"]]
dev_ids = pd.read_csv(DEV_SPLIT)[["par_id"]]

train_df = pcl_df.merge(train_ids, how="inner", on="par_id")
dev_df = pcl_df.merge(dev_ids, how="inner", on="par_id")

# ---- Task 2 spans (localized supervision) ----
# We only need the character offsets; we'll aggregate them into per-paragraph span_ranges.
spans_df = pd.read_csv(RAW_TASK2, sep="\t", header=None, skiprows=4)
spans_df.columns = [
    "par_id", "art_id", "text", "keyword", "country_code",
    "span_start", "span_finish", "span_text", "category", "n_annotators",
]

# Keep only valid span offsets.
spans_df = spans_df.dropna(subset=["span_start", "span_finish"])
spans_df["span_start"] = spans_df["span_start"].astype(int)
spans_df["span_finish"] = spans_df["span_finish"].astype(int)

span_ranges_by_par = (
    spans_df.groupby("par_id")[["span_start", "span_finish"]]
    .apply(lambda d: list(map(tuple, d.values.tolist())))
    .rename("span_ranges")
    .reset_index()
)

train_df = train_df.merge(span_ranges_by_par, on="par_id", how="left")
dev_df = dev_df.merge(span_ranges_by_par, on="par_id", how="left")

# Replace missing spans with empty list (negatives will typically have no spans).
train_df["span_ranges"] = train_df["span_ranges"].apply(lambda x: x if isinstance(x, list) else [])
dev_df["span_ranges"] = dev_df["span_ranges"].apply(lambda x: x if isinstance(x, list) else [])

# Minimal sanity checks for the dataset class later
assert "text" in train_df.columns and "label_bin" in train_df.columns and "span_ranges" in train_df.columns

print("train_df:", train_df.shape, "| positives:", int(train_df["label_bin"].sum()))
print("dev_df:", dev_df.shape, "| positives:", int(dev_df["label_bin"].sum()))
print("Example span_ranges:", train_df.loc[train_df["label_bin"].idxmax(), "span_ranges"] if len(train_df) else None)

train_df: (8375, 8) | positives: 794
dev_df: (2094, 8) | positives: 199
Example span_ranges: [(157, 243)]


In [ ]:
# Integrity checks: train_df span_ranges vs spans_df + verify offsets vs Task1 substring
# - shows both (A) plain clamped substring and (B) length-capped substring:
#   end := min(end_clamped, start_clamped + len(span_text))

import pandas as pd
import numpy as np

def _to_str_safe(x) -> str:
    if x is None:
        return ""
    if isinstance(x, float) and np.isnan(x):
        return ""
    return str(x)

def _normalize_span_list(x):
    if isinstance(x, list):
        out = []
        for t in x:
            if isinstance(t, (tuple, list)) and len(t) == 2:
                out.append((int(t[0]), int(t[1])))
        return out
    return []

# ---- Clean Task2 ----
sp2 = spans_df.dropna(subset=["span_start", "span_finish"]).copy()
sp2["span_start"] = sp2["span_start"].astype(int)
sp2["span_finish"] = sp2["span_finish"].astype(int)
sp2["_t2_text"] = sp2["text"].map(_to_str_safe)
sp2["_t2_L"] = sp2["_t2_text"].map(len)
sp2["_span_text"] = sp2["span_text"].map(_to_str_safe)
sp2["_span_text_L"] = sp2["_span_text"].map(len)

# ---- Attach Task1 paragraph text by par_id ----
task1_text_by_par = pcl_df.set_index("par_id")["text"].map(_to_str_safe).to_dict()
sp2["_t1_text"] = sp2["par_id"].map(lambda pid: task1_text_by_par.get(pid, ""))
sp2["_t1_L"] = sp2["_t1_text"].map(len)

# ---- Quick: which reference do offsets fit? ----
in_bounds_t1 = (
    (sp2["span_start"] >= 0)
    & (sp2["span_finish"] >= 0)
    & (sp2["span_finish"] <= sp2["_t1_L"])
).mean()
in_bounds_t2 = (
    (sp2["span_start"] >= 0)
    & (sp2["span_finish"] >= 0)
    & (sp2["span_finish"] <= sp2["_t2_L"])
).mean()
print(f"Task2 offset in-bounds rate vs Task1 text: {in_bounds_t1:.4f}")
print(f"Task2 offset in-bounds rate vs Task2 text: {in_bounds_t2:.4f}")

def _clamp_span(L: int, s: int, e: int):
    s0, e0 = int(s), int(e)
    s1 = max(0, min(s0, L))
    e1 = max(0, min(e0, L))
    return s1, e1, (s1 != s0) or (e1 != e0)

def _substr_clamped(t: str, s: int, e: int):
    L = len(t)
    s1, e1, was_clamped = _clamp_span(L, s, e)
    if s1 >= e1:
        return "", s1, e1, was_clamped, False
    return t[s1:e1], s1, e1, was_clamped, True

def _substr_clamped_len_cap(t: str, s: int, e: int, span_text: str):
    """
    Same clamping as above, but also caps end to:
      end = min(end_clamped, start_clamped + len(span_text))
    This fixes most inclusive/exclusive quirks around trailing punctuation.
    """
    L = len(t)
    s1, e1, was_clamped = _clamp_span(L, s, e)
    # cap using span_text length (if empty, do nothing special)
    cap = s1 + len(span_text or "")
    e2 = min(e1, cap)

    # if cap makes it invalid, fall back to the plain clamped span
    if s1 >= e2:
        if s1 >= e1:
            return "", s1, e1, e2, was_clamped, True, False
        return t[s1:e1], s1, e1, e1, was_clamped, True, True

    return t[s1:e2], s1, e1, e2, was_clamped, True, True

# ---- Build substrings (plain clamped) ----
subs_plain = [
    _substr_clamped(t, s, e)
    for t, s, e in zip(
        sp2["_t1_text"].tolist(),
        sp2["span_start"].tolist(),
        sp2["span_finish"].tolist(),
    )
]
sp2["_t1_substr_plain"] = [x[0] for x in subs_plain]
sp2["_s_clamped"] = [x[1] for x in subs_plain]
sp2["_e_clamped"] = [x[2] for x in subs_plain]
sp2["_was_clamped"] = [x[3] for x in subs_plain]
sp2["_substr_ok_plain"] = [x[4] for x in subs_plain]

# ---- Build substrings (length-capped) ----
subs_cap = [
    _substr_clamped_len_cap(t, s, e, st)
    for t, s, e, st in zip(
        sp2["_t1_text"].tolist(),
        sp2["span_start"].tolist(),
        sp2["span_finish"].tolist(),
        sp2["_span_text"].tolist(),
    )
]
sp2["_t1_substr_cap"] = [x[0] for x in subs_cap]
sp2["_e_after_len_cap"] = [x[3] for x in subs_cap]   # the end actually used after len-cap
sp2["_substr_ok_cap"] = [x[5] for x in subs_cap]
sp2["_used_len_cap"] = [x[6] for x in subs_cap]

# Compare (strip whitespace only)
span_match_plain = (sp2["_t1_substr_plain"].map(str.strip) == sp2["_span_text"].map(str.strip))
span_match_cap = (sp2["_t1_substr_cap"].map(str.strip) == sp2["_span_text"].map(str.strip))

print(f"Task2 span_text matches Task1 substring @ offsets (plain clamp): {span_match_plain.mean():.4f}")
print(f"Task2 span_text matches Task1 substring @ offsets (len-capped):  {span_match_cap.mean():.4f}")

# Show remaining mismatches using the better (len-capped) substring
bad_examples = sp2.loc[
    ~span_match_cap,
    [
        "par_id", "span_start", "span_finish",
        "_s_clamped", "_e_clamped", "_e_after_len_cap", "_was_clamped", "_used_len_cap",
        "_t1_L", "_t2_L",
        "_span_text", "_t1_substr_cap",
    ],
]
print(f"Remaining mismatches after len-cap: {len(bad_examples)} / {len(sp2)}")
if len(bad_examples):
    display(bad_examples.head(40))

# ---- Build lookup: par_id -> set of (start, finish) from Task2 ----
spans_lookup = (
    sp2.groupby("par_id")[["span_start", "span_finish"]]
    .apply(lambda d: set(map(tuple, d.to_numpy().tolist())))
    .to_dict()
)

def validate_train_df(df: pd.DataFrame, name: str, assert_negatives_empty: bool = False, assert_in_bounds: bool = True):
    print(f"\n== Validating {name} ==")
    assert {"par_id", "text", "label_bin", "span_ranges"}.issubset(df.columns)
    print("shape:", df.shape)

    spans_col = df["span_ranges"].apply(_normalize_span_list)

    # (A) every span_ranges pair must exist in spans_df for that par_id
    missing_pairs = []
    for par_id, span_list in zip(df["par_id"].tolist(), spans_col.tolist()):
        if not span_list:
            continue
        valid = spans_lookup.get(par_id, set())
        for s, e in span_list:
            if (s, e) not in valid:
                missing_pairs.append((par_id, s, e))
    print("span_ranges pairs missing from spans_df (same par_id):", len(missing_pairs))
    if missing_pairs[:10]:
        print("examples:", missing_pairs[:10])

    # (B) bounds vs Task1 paragraph text length (strict)
    out_of_bounds = []
    out_of_bounds_off_by_one = []
    for par_id, t1, span_list in zip(df["par_id"].tolist(), df["text"].tolist(), spans_col.tolist()):
        t = _to_str_safe(t1)
        L = len(t)
        for s, e in span_list:
            if s < 0 or e < 0 or s >= e:
                out_of_bounds.append((par_id, s, e, L))
                continue
            if s > L or e > L:
                out_of_bounds.append((par_id, s, e, L))
                if (0 <= s <= L) and (e == L + 1):
                    out_of_bounds_off_by_one.append((par_id, s, e, L))

    print("span_ranges out-of-bounds vs train_df text length:", len(out_of_bounds))
    if len(out_of_bounds_off_by_one):
        print("...of which appear to be simple end==L+1 off-by-one:", len(out_of_bounds_off_by_one))
    if out_of_bounds[:10]:
        print("examples (par_id, s, e, L):", out_of_bounds[:10])

    # (C) negatives empty span_ranges
    neg_mask = df["label_bin"].astype(int).eq(0)
    neg_with_spans = df.loc[neg_mask].copy()
    neg_with_spans["_sp"] = spans_col.loc[neg_mask].tolist()
    neg_with_spans = neg_with_spans[neg_with_spans["_sp"].apply(len) > 0][["par_id", "label_bin", "_sp"]]
    print("label_bin==0 rows with non-empty span_ranges:", len(neg_with_spans))
    if len(neg_with_spans):
        display(neg_with_spans.head(20))

    # Assertions (optional)
    assert len(missing_pairs) == 0, "Found span_ranges pairs not present in spans_df for that par_id"
    if assert_in_bounds:
        assert len(out_of_bounds) == 0, "Found span_ranges out of bounds vs Task1 text length"
    if assert_negatives_empty:
        assert len(neg_with_spans) == 0, "Found label_bin==0 rows with non-empty span_ranges"

# NOTE: set assert_in_bounds=False if you want to keep going while inspecting off-by-one cases.
validate_train_df(train_df, "train_df", assert_negatives_empty=False, assert_in_bounds=True)
validate_train_df(dev_df, "dev_df", assert_negatives_empty=False, assert_in_bounds=True)

/home/joshua_killa/.pyenv/versions/pcl-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel

class TokenCLSModel(nn.Module):
    def __init__(self, model_name, lambda_token=0.3, top_k=3):
        super().__init__()

        # Online first, then local cache fallback
        try:
            self.encoder = AutoModel.from_pretrained(model_name)
        except Exception as e:
            print(f"Model load failed ({type(e).__name__}: {e}). Trying local cache...")
            self.encoder = AutoModel.from_pretrained(model_name, local_files_only=True)

        hidden = self.encoder.config.hidden_size

        self.token_head = nn.Linear(hidden, 1)
        self.paragraph_head = nn.Linear(hidden + 1, 1)

        self.lambda_token = float(lambda_token)
        self.top_k = int(top_k)

    def forward(
        self,
        input_ids,
        attention_mask,
        token_type_ids=None,
        token_labels=None,
        token_loss_mask=None,
        paragraph_label=None,
    ):
        # Some backbones ignore token_type_ids; some accept it.
        # Try passing it; if unsupported, fall back.
        try:
            outputs = self.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
            )
        except TypeError:
            outputs = self.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask,
            )

        hidden = outputs.last_hidden_state  # (B,T,H)

        # keep dtype consistent with heads (prevents dtype mismatch issues)
        hidden = hidden.to(dtype=self.token_head.weight.dtype)

        cls = hidden[:, 0]  # (B,H)
        token_logits = self.token_head(hidden).squeeze(-1)  # (B,T)

        # safe token mask for pooling + token-loss
        if token_loss_mask is None:
            token_loss_mask = attention_mask == 1
        token_loss_mask = token_loss_mask.to(dtype=torch.bool)

        # pooling: avoid huge negatives that can become -inf in mixed precision
        token_logits_for_pool = token_logits.masked_fill(~token_loss_mask, -1e4)
        topk_vals, _ = torch.topk(token_logits_for_pool, k=self.top_k, dim=1)
        pooled = topk_vals.mean(dim=1, keepdim=True)  # (B,1)

        fused = torch.cat([cls, pooled], dim=1)  # (B,H+1)
        paragraph_logit = self.paragraph_head(fused).squeeze(-1)  # (B,)

        loss = None
        loss_par = None
        loss_tok = None

        if paragraph_label is not None:
            # Compute BCE losses in FP32 for stability (important under AMP)
            loss_par = F.binary_cross_entropy_with_logits(
                paragraph_logit.float(),
                paragraph_label.float(),
            )

            tok_logits_flat = token_logits[token_loss_mask]
            if tok_logits_flat.numel() == 0:
                loss_tok = torch.zeros((), device=token_logits.device, dtype=torch.float32)
            else:
                tok_labels_flat = token_labels[token_loss_mask].float()
                loss_tok = F.binary_cross_entropy_with_logits(
                    tok_logits_flat.float(),
                    tok_labels_flat,
                )

            loss = loss_par + self.lambda_token * loss_tok

        return {
            "loss": loss,
            "loss_par": loss_par,
            "loss_tok": loss_tok,
            "paragraph_logit": paragraph_logit,
            "token_logits": token_logits,
        }

In [ ]:
from torch.utils.data import DataLoader
from torch.optim import AdamW
from tqdm import tqdm
import torch
import numpy as np
from sklearn.metrics import f1_score

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def _amp_settings(model_key: str, model_name: str):
    key = (model_key or "").lower()
    name = (model_name or "").lower()

    if ("albert" in key) or ("albert" in name):
        enabled = torch.cuda.is_available()
        return enabled, torch.float16, True  # FP16 + GradScaler

    if ("deberta" in key) or ("deberta" in name):
        enabled = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
        return enabled, torch.bfloat16, False  # BF16, no GradScaler

    return False, None, False

USE_AMP, AMP_DTYPE, NEEDS_SCALER = _amp_settings(MODEL_KEY, MODEL_NAME)
scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and NEEDS_SCALER))

print(
    f"DEVICE={DEVICE} | backbone={MODEL_KEY} | amp={USE_AMP} | amp_dtype={AMP_DTYPE} | "
    f"scaler={bool(scaler.is_enabled())}"
)

model = TokenCLSModel(MODEL_NAME, lambda_token=0.3, top_k=3).to(DEVICE)

train_dataset = PCLTokenDataset(train_df)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

# separate loaders for F1 computation (no shuffle)
train_eval_loader = DataLoader(train_dataset, batch_size=64, shuffle=False)
dev_dataset = PCLTokenDataset(dev_df)
dev_eval_loader = DataLoader(dev_dataset, batch_size=64, shuffle=False)

optimizer = AdamW(model.parameters(), lr=2e-5)
EPOCHS = 7

def f1_on_loader(loader) -> float:
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}

            if USE_AMP:
                with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                    out = model(**batch)
            else:
                out = model(**batch)

            all_logits.append(out["paragraph_logit"].detach().float().cpu())
            all_labels.append(batch["paragraph_label"].detach().float().cpu())

    logits = torch.cat(all_logits).numpy()
    labels = torch.cat(all_labels).numpy().astype(int)

    probs = 1.0 / (1.0 + np.exp(-logits))
    preds = (probs > 0.5).astype(int)
    return f1_score(labels, preds, pos_label=1)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for batch in tqdm(train_loader):
        optimizer.zero_grad(set_to_none=True)
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        if USE_AMP:
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                out = model(**batch)
                loss = out["loss"]
        else:
            out = model(**batch)
            loss = out["loss"]

        if torch.isnan(loss) or torch.isinf(loss):
            raise RuntimeError("Loss became NaN/Inf. Inspect batch / masks / logits.")

        if scaler.is_enabled():
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        total_loss += float(loss.detach().cpu())

    train_f1 = f1_on_loader(train_eval_loader)
    dev_f1 = f1_on_loader(dev_eval_loader)

    print(
        f"Epoch {epoch} | Loss: {total_loss / len(train_loader):.4f} | "
        f"train_f1@0.5: {train_f1:.4f} | dev_f1@0.5: {dev_f1:.4f} | "
        f"backbone={MODEL_KEY} | amp={USE_AMP} | dtype={AMP_DTYPE}"
    )

/tmp/ipykernel_263150/1771181930.py:25: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and NEEDS_SCALER))


DEVICE=cuda | backbone=albert | amp=True | amp_dtype=torch.float16 | scaler=True


Loading weights: 100%|██████████| 25/25 [00:00<00:00, 376.69it/s, Materializing param=pooler.weight]                                                             
AlbertModel LOAD REPORT from: albert-base-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
predictions.bias             | UNEXPECTED |  | 
predictions.LayerNorm.weight | UNEXPECTED |  | 
predictions.dense.weight     | UNEXPECTED |  | 
predictions.LayerNorm.bias   | UNEXPECTED |  | 
predictions.dense.bias       | UNEXPECTED |  | 
predictions.decoder.bias     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
  7%|▋         | 35/524 [00:10<02:22,  3.44it/s]


KeyboardInterrupt: 

In [ ]:
from sklearn.metrics import f1_score
import numpy as np

def evaluate(model, dataset):
    loader = DataLoader(dataset, batch_size=32)
    model.eval()

    all_logits = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            out = model(**batch)
            logits = out["paragraph_logit"]
            all_logits.append(logits.cpu())
            all_labels.append(batch["paragraph_label"].cpu())

    logits = torch.cat(all_logits).numpy()
    labels = torch.cat(all_labels).numpy()

    probs = 1 / (1 + np.exp(-logits))

    best_f1 = 0
    best_thresh = 0.5

    for t in np.linspace(0.1, 0.9, 81):
        preds = (probs > t).astype(int)
        f1 = f1_score(labels, preds)
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = t

    return best_f1, best_thresh